# Google Colab - Entrainement final H2S : Random Forest + LSTM

Ce notebook est la version **Google Colab** du protocole final d'entrainement.

Il applique la methodologie complete :

- chargement de `DATASET01.csv` par upload Colab ou Google Drive ;
- nettoyage des donnees ;
- traitement des valeurs manquantes ;
- suppression des doublons ;
- detection des valeurs aberrantes ;
- correction des incoherences ;
- fusion des capteurs H2S ;
- ajout de la temperature et de l'humidite ;
- equilibrage des classes `0`, `1`, `2` ;
- encodage de la cible ;
- standardisation ;
- entrainement Random Forest ;
- entrainement LSTM ;
- evaluation complete des metriques ;
- analyse du surapprentissage ;
- sauvegarde et telechargement des resultats.

Le modele reste centre sur le gaz **H2S**. Les quatre capteurs historiques sont fusionnes en une mesure principale : `C_H2S_fusion_ppm`.


## 1. Preparation de l'environnement Colab

Dans Google Colab, les bibliotheques principales sont generalement deja installees. Si une dependance manque, decommenter la ligne d'installation.


In [ ]:
# Dans Google Colab, cette ligne peut etre activee si necessaire :
# !pip -q install pandas numpy scikit-learn tensorflow joblib matplotlib

import os
import sys
import json
import random
import zipfile
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, log_loss
)
from sklearn.model_selection import train_test_split
import joblib

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization, Bidirectional
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
except Exception as exc:
    tf = None
    print("TensorFlow indisponible :", exc)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if tf is not None:
    tf.keras.utils.set_random_seed(SEED)
    tf.random.set_seed(SEED)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

print("Environnement pret.")
if tf is not None:
    print("TensorFlow :", tf.__version__)


## 2. Charger DATASET01.csv dans Colab

Trois options sont possibles :

1. Mettre `DATASET01.csv` dans `/content` ;
2. utiliser l'upload manuel Colab ;
3. utiliser Google Drive en mettant `USE_GOOGLE_DRIVE = True`.


In [ ]:
USE_GOOGLE_DRIVE = False
DRIVE_DATASET_PATH = "/content/drive/MyDrive/DATASET01.csv"

def find_or_upload_dataset() -> Path:
    candidates = [
        Path("/content/DATASET01.csv"),
        Path("DATASET01.csv"),
        # Chemin local garde uniquement pour validation hors Colab.
        Path(r"D:\TFC_2026 (2)\Mr. RUPHIN\final\CORRECTION\MODELE\DATASET01.csv"),
    ]

    if USE_GOOGLE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            drive_path = Path(DRIVE_DATASET_PATH)
            if drive_path.exists():
                return drive_path
            raise FileNotFoundError(f"Fichier absent dans Google Drive : {drive_path}")
        except Exception as exc:
            raise RuntimeError(f"Impossible de charger depuis Google Drive : {exc}") from exc

    for candidate in candidates:
        if candidate.exists():
            return candidate

    try:
        from google.colab import files
        print("Selectionner DATASET01.csv")
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("Aucun fichier CSV uploade.")
        return Path(csv_files[0])
    except Exception as exc:
        raise FileNotFoundError("DATASET01.csv introuvable. Uploader le fichier dans Colab.") from exc

DATASET_PATH = find_or_upload_dataset()
raw_data = pd.read_csv(DATASET_PATH)
print("Dataset utilise :", DATASET_PATH)
print("Dimensions brutes :", raw_data.shape)
display(raw_data.head())


## 3. Parametres du projet


In [ ]:
TIME_COL = "Time[sec]"
SENSOR_COLS = ["Sensor1[ppm]", "Sensor2[ppm]", "Sensor3[ppm]", "Sensor4[ppm]"]
HUMIDITY_COL = "Humidity[%]"
TEMPERATURE_COL = "Temperature[C]"
TRUE_CONCENTRATION_COL = "True_concentration[ppm]"
TARGET_COL = "Labels"

FUSION_COL = "C_H2S_fusion_ppm"
TARGET_NAMES = {0: "NORMAL", 1: "MODERE", 2: "DANGEREUX"}

MAX_MISSING_RATIO_COLUMN = 0.40
MAX_MISSING_RATIO_ROW = 0.50
TEMPERATURE_RANGE = (-40, 85)
HUMIDITY_RANGE = (0, 100)

BALANCE_ROWS_PER_CLASS = 20_000

RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 8
RF_MIN_SAMPLES_LEAF = 2

LSTM_WINDOW = 90
LSTM_HORIZON = 10
LSTM_STRIDE = 16
MAX_LSTM_SEQUENCES_PER_CLASS = 8_000
LSTM_EPOCHS = 30
LSTM_BATCH_SIZE = 128

IN_COLAB = "google.colab" in sys.modules
COLAB_ROOT = Path("/content") if IN_COLAB else Path.cwd()
OUTPUT_DIR = COLAB_ROOT / "outputs_h2s_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Dossier de sortie :", OUTPUT_DIR)


## 4. Audit des valeurs manquantes


In [ ]:
expected_columns = [TIME_COL, *SENSOR_COLS, HUMIDITY_COL, TEMPERATURE_COL, TRUE_CONCENTRATION_COL, TARGET_COL]
missing_expected = [column for column in expected_columns if column not in raw_data.columns]
if missing_expected:
    raise ValueError(f"Colonnes attendues absentes : {missing_expected}")

missing_report = pd.DataFrame({
    "colonne": raw_data.columns,
    "valeurs_manquantes": raw_data.isna().sum().values,
    "ratio_manquant": raw_data.isna().mean().values,
}).sort_values("ratio_manquant", ascending=False)
display(missing_report)

columns_to_drop = missing_report.loc[missing_report["ratio_manquant"] > MAX_MISSING_RATIO_COLUMN, "colonne"].tolist()
columns_to_drop = [column for column in columns_to_drop if column not in expected_columns]
print("Colonnes supprimees pour trop de valeurs manquantes :", columns_to_drop)


## 5. Nettoyage des donnees

Traitements appliques :

- conversion numerique ;
- suppression des lignes trop incompletes ;
- conservation des labels `0`, `1`, `2` ;
- correction des valeurs physiques impossibles ;
- imputation par propagation temporelle puis mediane ;
- suppression des doublons.


In [ ]:
clean_data = raw_data.drop(columns=columns_to_drop, errors="ignore")[expected_columns].copy()
rows_initial = len(clean_data)

for column in expected_columns:
    clean_data[column] = pd.to_numeric(clean_data[column], errors="coerce")
clean_data = clean_data.replace([np.inf, -np.inf], np.nan)

row_missing_ratio = clean_data.isna().mean(axis=1)
rows_too_missing = int((row_missing_ratio > MAX_MISSING_RATIO_ROW).sum())
clean_data = clean_data.loc[row_missing_ratio <= MAX_MISSING_RATIO_ROW].copy()

clean_data = clean_data.dropna(subset=[TARGET_COL]).copy()
clean_data = clean_data[clean_data[TARGET_COL].isin([0, 1, 2])].copy()
clean_data[TARGET_COL] = clean_data[TARGET_COL].astype(int)
clean_data = clean_data.sort_values(TIME_COL).reset_index(drop=True)

for sensor in SENSOR_COLS:
    clean_data.loc[clean_data[sensor] < 0, sensor] = np.nan
clean_data.loc[~clean_data[TEMPERATURE_COL].between(*TEMPERATURE_RANGE), TEMPERATURE_COL] = np.nan
clean_data.loc[~clean_data[HUMIDITY_COL].between(*HUMIDITY_RANGE), HUMIDITY_COL] = np.nan

numeric_to_impute = [*SENSOR_COLS, TEMPERATURE_COL, HUMIDITY_COL, TRUE_CONCENTRATION_COL]
clean_data[numeric_to_impute] = clean_data[numeric_to_impute].ffill().bfill()
for column in numeric_to_impute:
    clean_data[column] = clean_data[column].fillna(clean_data[column].median())

duplicates_before = int(clean_data.duplicated().sum())
clean_data = clean_data.drop_duplicates().reset_index(drop=True)

cleaning_summary = pd.DataFrame({
    "traitement": [
        "lignes_initiales",
        "lignes_supprimees_pour_trop_de_manquants",
        "doublons_supprimes",
        "lignes_finales",
        "valeurs_nulles_restantes",
    ],
    "valeur": [
        rows_initial,
        rows_too_missing,
        duplicates_before,
        len(clean_data),
        int(clean_data.isna().sum().sum()),
    ],
})
display(cleaning_summary)


## 6. Detection des valeurs aberrantes


In [ ]:
def iqr_outlier_report(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for column in columns:
        q1 = frame[column].quantile(0.25)
        q3 = frame[column].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = int(((frame[column] < lower) | (frame[column] > upper)).sum())
        rows.append({
            "variable": column,
            "q1": q1,
            "q3": q3,
            "borne_basse_iqr": lower,
            "borne_haute_iqr": upper,
            "outliers_detectes": count,
            "decision": "conserver si plausible / supprimer seulement impossible physiquement",
        })
    return pd.DataFrame(rows)

outlier_report = iqr_outlier_report(clean_data, [*SENSOR_COLS, TEMPERATURE_COL, HUMIDITY_COL])
display(outlier_report.round(3))


## 7. Fusion H2S et variables d'apprentissage

On fusionne les quatre capteurs H2S, puis on ajoute des statistiques de fusion, la temperature et l'humidite.


In [ ]:
model_data = clean_data.copy()
model_data[FUSION_COL] = model_data[SENSOR_COLS].mean(axis=1)
model_data["H2S_std_capteurs"] = model_data[SENSOR_COLS].std(axis=1)
model_data["H2S_min_capteurs"] = model_data[SENSOR_COLS].min(axis=1)
model_data["H2S_max_capteurs"] = model_data[SENSOR_COLS].max(axis=1)
model_data["H2S_amplitude_capteurs"] = model_data["H2S_max_capteurs"] - model_data["H2S_min_capteurs"]

h2s = model_data[FUSION_COL]
for window in [3, 5, 15, 30, 60]:
    model_data[f"H2S_diff_{window}"] = h2s.diff(window).fillna(0)
    model_data[f"H2S_moy_mobile_{window}"] = h2s.rolling(window, min_periods=1).mean()
    model_data[f"H2S_std_mobile_{window}"] = h2s.rolling(window, min_periods=1).std().fillna(0)

RF_FEATURES = [
    FUSION_COL,
    "H2S_std_capteurs",
    "H2S_min_capteurs",
    "H2S_max_capteurs",
    "H2S_amplitude_capteurs",
    "H2S_diff_5",
    "H2S_moy_mobile_5",
    "H2S_std_mobile_5",
    "H2S_diff_30",
    "H2S_moy_mobile_30",
    "H2S_std_mobile_30",
    TEMPERATURE_COL,
    HUMIDITY_COL,
]
LSTM_FEATURES = [
    *SENSOR_COLS,
    FUSION_COL,
    "H2S_std_capteurs",
    "H2S_min_capteurs",
    "H2S_max_capteurs",
    "H2S_amplitude_capteurs",
    "H2S_diff_3",
    "H2S_moy_mobile_3",
    "H2S_std_mobile_3",
    "H2S_diff_5",
    "H2S_moy_mobile_5",
    "H2S_std_mobile_5",
    "H2S_diff_15",
    "H2S_moy_mobile_15",
    "H2S_std_mobile_15",
    "H2S_diff_30",
    "H2S_moy_mobile_30",
    "H2S_std_mobile_30",
    "H2S_diff_60",
    "H2S_moy_mobile_60",
    "H2S_std_mobile_60",
    TEMPERATURE_COL,
    HUMIDITY_COL,
]

display(model_data[RF_FEATURES + [TARGET_COL]].head())
print("Variables Random Forest :", RF_FEATURES)
print("Variables LSTM :", LSTM_FEATURES)


## 8. Encodage de la cible et equilibrage des classes


In [ ]:
label_encoder = LabelEncoder()
model_data["target_encoded"] = label_encoder.fit_transform(model_data[TARGET_COL].astype(int))
print("Classes encodees :", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

def balance_dataframe(frame: pd.DataFrame, target_col: str, rows_per_class: int, random_state: int = SEED) -> pd.DataFrame:
    balanced_parts = []
    for target_value, group in frame.groupby(target_col):
        n_rows = min(len(group), rows_per_class)
        balanced_parts.append(group.sample(n_rows, random_state=random_state + int(target_value)))
    return pd.concat(balanced_parts, ignore_index=True).sample(frac=1, random_state=random_state).reset_index(drop=True)

class_before = model_data[TARGET_COL].value_counts().sort_index().rename_axis("classe").reset_index(name="avant_equilibrage")
balanced_data = balance_dataframe(model_data, TARGET_COL, BALANCE_ROWS_PER_CLASS)
class_after = balanced_data[TARGET_COL].value_counts().sort_index().rename_axis("classe").reset_index(name="apres_equilibrage")
display(class_before.merge(class_after, on="classe", how="outer"))

plt.figure(figsize=(7, 4))
plt.bar(class_after["classe"].astype(str), class_after["apres_equilibrage"], color=["#27ae60", "#f2994a", "#eb5757"])
plt.title("Distribution apres equilibrage des classes")
plt.xlabel("Classe")
plt.ylabel("Nombre d'observations")
plt.tight_layout()
plt.show()


## 9. Entrainement Random Forest


In [ ]:
X = balanced_data[RF_FEATURES].copy()
y = balanced_data["target_encoded"].astype(int).copy()

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1875, random_state=SEED, stratify=y_train_full)

display(pd.DataFrame({"split": ["train", "validation", "test"], "lignes": [len(X_train), len(X_val), len(X_test)]}))

rf_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features="sqrt",
        bootstrap=True,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=1,
        oob_score=True,
    )),
])
rf_pipeline.fit(X_train, y_train)
rf_model = rf_pipeline.named_steps["model"]
print("OOB score :", round(float(rf_model.oob_score_), 4))


## 10. Evaluation complete Random Forest


In [ ]:
def evaluate_classifier(model, X_part, y_part, split_name: str):
    pred = model.predict(X_part)
    proba = model.predict_proba(X_part) if hasattr(model, "predict_proba") else None
    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_part, pred),
        "balanced_accuracy": balanced_accuracy_score(y_part, pred),
        "precision_macro": precision_score(y_part, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_part, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_part, pred, average="macro"),
        "f1_weighted": f1_score(y_part, pred, average="weighted"),
    }
    if proba is not None:
        metrics["log_loss"] = log_loss(y_part, proba, labels=[0, 1, 2])
        try:
            metrics["roc_auc_ovr_macro"] = roc_auc_score(y_part, proba, multi_class="ovr", average="macro")
        except Exception:
            metrics["roc_auc_ovr_macro"] = np.nan
    report = classification_report(
        y_part, pred, labels=[0, 1, 2],
        target_names=[TARGET_NAMES[i] for i in [0, 1, 2]],
        output_dict=True, zero_division=0,
    )
    cm = confusion_matrix(y_part, pred, labels=[0, 1, 2])
    return metrics, report, cm, pred

rf_train_metrics, rf_train_report, rf_train_cm, rf_train_pred = evaluate_classifier(rf_pipeline, X_train, y_train, "train")
rf_val_metrics, rf_val_report, rf_val_cm, rf_val_pred = evaluate_classifier(rf_pipeline, X_val, y_val, "validation")
rf_test_metrics, rf_test_report, rf_test_cm, rf_test_pred = evaluate_classifier(rf_pipeline, X_test, y_test, "test")

rf_metrics_df = pd.DataFrame([rf_train_metrics, rf_val_metrics, rf_test_metrics])
display(rf_metrics_df.round(4))
display(pd.DataFrame(rf_test_report).T.round(4))
display(pd.DataFrame(rf_test_cm, index=["reel_" + TARGET_NAMES[i] for i in [0,1,2]], columns=["pred_" + TARGET_NAMES[i] for i in [0,1,2]]))


## 11. Graphiques Random Forest et analyse du surapprentissage


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(rf_test_cm, cmap="Blues")
axes[0].set_title("Matrice de confusion - Random Forest")
axes[0].set_xticks([0, 1, 2], [TARGET_NAMES[i] for i in [0,1,2]], rotation=25)
axes[0].set_yticks([0, 1, 2], [TARGET_NAMES[i] for i in [0,1,2]])
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, int(rf_test_cm[i, j]), ha="center", va="center")
axes[0].set_xlabel("Prediction")
axes[0].set_ylabel("Classe reelle")

importance = pd.Series(rf_model.feature_importances_, index=RF_FEATURES).sort_values()
axes[1].barh(importance.index, importance.values, color="#1f77b4")
axes[1].set_title("Importance des variables")
axes[1].set_xlabel("Importance")
plt.tight_layout()
plt.show()

def interpret_overfitting(train_acc: float, test_acc: float) -> str:
    gap = train_acc - test_acc
    if train_acc >= 0.90 and test_acc >= 0.90 and abs(gap) <= 0.05:
        return "Bon modele : les performances train et test sont proches."
    if train_acc >= 0.95 and gap > 0.15:
        return "Surapprentissage : le modele memorise trop les donnees d'entrainement."
    if train_acc < 0.75 and test_acc < 0.75:
        return "Sous-apprentissage : le modele n'apprend pas suffisamment."
    return "Modele a surveiller : verifier l'ecart train/test et les courbes."

overfit_rf = pd.DataFrame([{
    "modele": "Random Forest",
    "train_accuracy": rf_train_metrics["accuracy"],
    "test_accuracy": rf_test_metrics["accuracy"],
    "ecart_train_test": rf_train_metrics["accuracy"] - rf_test_metrics["accuracy"],
    "interpretation": interpret_overfitting(rf_train_metrics["accuracy"], rf_test_metrics["accuracy"]),
}])
display(overfit_rf.round(4))


## 12. Preparation des sequences LSTM


In [ ]:
if tf is None:
    raise ImportError("TensorFlow est requis pour entrainer le LSTM.")

lstm_source = model_data[[TIME_COL, *LSTM_FEATURES, TARGET_COL]].sort_values(TIME_COL).reset_index(drop=True).copy()
lstm_source["future_target"] = lstm_source[TARGET_COL].shift(-LSTM_HORIZON)
lstm_source = lstm_source.dropna().reset_index(drop=True)
lstm_source["future_target"] = lstm_source["future_target"].astype(int)

feature_scaler = StandardScaler()
scaled_features = feature_scaler.fit_transform(lstm_source[LSTM_FEATURES])

def make_lstm_sequences(feature_array: np.ndarray, target_values: np.ndarray, window: int, stride: int):
    X_seq, y_seq = [], []
    for start in range(0, max(0, len(feature_array) - window), stride):
        end = start + window
        X_seq.append(feature_array[start:end])
        y_seq.append(target_values[end - 1])
    return np.asarray(X_seq, dtype="float32"), np.asarray(y_seq, dtype="int64")

X_sequences, y_sequences = make_lstm_sequences(scaled_features, lstm_source["future_target"].values, LSTM_WINDOW, LSTM_STRIDE)
sequence_data = pd.DataFrame({"sequence_index": np.arange(len(y_sequences)), "target": y_sequences})

sequence_parts = []
min_sequence_count = sequence_data["target"].value_counts().min()
sequence_count_per_class = min(int(min_sequence_count), MAX_LSTM_SEQUENCES_PER_CLASS)
for target_value, group in sequence_data.groupby("target"):
    sequence_parts.append(group.sample(sequence_count_per_class, random_state=SEED + int(target_value)))
balanced_sequences = pd.concat(sequence_parts, ignore_index=True).sample(frac=1, random_state=SEED)
selected_idx = balanced_sequences["sequence_index"].values
X_lstm = X_sequences[selected_idx]
y_lstm = y_sequences[selected_idx]

X_lstm_train_full, X_lstm_test, y_lstm_train_full, y_lstm_test = train_test_split(X_lstm, y_lstm, test_size=0.20, random_state=SEED, stratify=y_lstm)
X_lstm_train, X_lstm_val, y_lstm_train, y_lstm_val = train_test_split(X_lstm_train_full, y_lstm_train_full, test_size=0.1875, random_state=SEED, stratify=y_lstm_train_full)

display(sequence_data["target"].map(TARGET_NAMES).value_counts().rename_axis("classe_avant").reset_index(name="nombre"))
display(pd.Series(y_lstm).map(TARGET_NAMES).value_counts().rename_axis("classe_apres_equilibrage").reset_index(name="nombre"))
display(pd.DataFrame({"split": ["train", "validation", "test"], "sequences": [len(X_lstm_train), len(X_lstm_val), len(X_lstm_test)]}))
print("Forme X_lstm_train :", X_lstm_train.shape)


## 13. Entrainement et evaluation LSTM


In [ ]:
lstm_model = Sequential([
    Input(shape=(LSTM_WINDOW, len(LSTM_FEATURES))),
    Bidirectional(LSTM(64, return_sequences=True, activation="tanh")),
    BatchNormalization(),
    Dropout(0.25),
    LSTM(32, activation="tanh"),
    BatchNormalization(),
    Dropout(0.25),
    Dense(32, activation="relu"),
    Dropout(0.15),
    Dense(3, activation="softmax"),
])
lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history = lstm_model.fit(
    X_lstm_train,
    y_lstm_train,
    validation_data=(X_lstm_val, y_lstm_val),
    epochs=LSTM_EPOCHS,
    batch_size=LSTM_BATCH_SIZE,
    callbacks=[
        EarlyStopping(monitor="val_accuracy", patience=8, mode="max", restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_accuracy", patience=4, mode="max", factor=0.5, min_lr=1e-5),
    ],
    verbose=1,
)
history_df = pd.DataFrame(history.history)

def evaluate_lstm(model, X_part, y_part, split_name: str):
    proba = model.predict(X_part, verbose=0)
    pred = np.argmax(proba, axis=1)
    metrics = {
        "split": split_name,
        "accuracy": accuracy_score(y_part, pred),
        "balanced_accuracy": balanced_accuracy_score(y_part, pred),
        "precision_macro": precision_score(y_part, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_part, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_part, pred, average="macro"),
        "f1_weighted": f1_score(y_part, pred, average="weighted"),
        "log_loss": log_loss(y_part, proba, labels=[0, 1, 2]),
    }
    try:
        metrics["roc_auc_ovr_macro"] = roc_auc_score(y_part, proba, multi_class="ovr", average="macro")
    except Exception:
        metrics["roc_auc_ovr_macro"] = np.nan
    report = classification_report(y_part, pred, labels=[0, 1, 2], target_names=[TARGET_NAMES[i] for i in [0, 1, 2]], output_dict=True, zero_division=0)
    cm = confusion_matrix(y_part, pred, labels=[0, 1, 2])
    return metrics, report, cm, pred

lstm_train_metrics, lstm_train_report, lstm_train_cm, lstm_train_pred = evaluate_lstm(lstm_model, X_lstm_train, y_lstm_train, "train")
lstm_val_metrics, lstm_val_report, lstm_val_cm, lstm_val_pred = evaluate_lstm(lstm_model, X_lstm_val, y_lstm_val, "validation")
lstm_test_metrics, lstm_test_report, lstm_test_cm, lstm_test_pred = evaluate_lstm(lstm_model, X_lstm_test, y_lstm_test, "test")

lstm_metrics_df = pd.DataFrame([lstm_train_metrics, lstm_val_metrics, lstm_test_metrics])
display(lstm_metrics_df.round(4))
display(pd.DataFrame(lstm_test_report).T.round(4))
display(pd.DataFrame(lstm_test_cm, index=["reel_" + TARGET_NAMES[i] for i in [0,1,2]], columns=["pred_" + TARGET_NAMES[i] for i in [0,1,2]]))


## 14. Courbes LSTM et analyse du surapprentissage


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
history_df[["loss", "val_loss"]].plot(ax=axes[0])
axes[0].set_title("LSTM - Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1])
axes[1].set_title("LSTM - Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[2].imshow(lstm_test_cm, cmap="Purples")
axes[2].set_title("Matrice de confusion LSTM")
axes[2].set_xticks([0,1,2], [TARGET_NAMES[i] for i in [0,1,2]], rotation=25)
axes[2].set_yticks([0,1,2], [TARGET_NAMES[i] for i in [0,1,2]])
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, int(lstm_test_cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()

overfit_lstm = pd.DataFrame([{
    "modele": "LSTM",
    "train_accuracy": lstm_train_metrics["accuracy"],
    "test_accuracy": lstm_test_metrics["accuracy"],
    "ecart_train_test": lstm_train_metrics["accuracy"] - lstm_test_metrics["accuracy"],
    "interpretation": interpret_overfitting(lstm_train_metrics["accuracy"], lstm_test_metrics["accuracy"]),
}])
display(overfit_lstm.round(4))


## 15. Synthese, sauvegarde et telechargement


In [ ]:
summary = pd.DataFrame([
    {
        "modele": "Random Forest",
        "objectif": "Classification instantanee du risque H2S",
        "train_accuracy": rf_train_metrics["accuracy"],
        "validation_accuracy": rf_val_metrics["accuracy"],
        "test_accuracy": rf_test_metrics["accuracy"],
        "test_f1_macro": rf_test_metrics["f1_macro"],
        "etat_overfitting": overfit_rf.loc[0, "interpretation"],
    },
    {
        "modele": "LSTM",
        "objectif": "Prediction temporelle de la classe future",
        "train_accuracy": lstm_train_metrics["accuracy"],
        "validation_accuracy": lstm_val_metrics["accuracy"],
        "test_accuracy": lstm_test_metrics["accuracy"],
        "test_f1_macro": lstm_test_metrics["f1_macro"],
        "etat_overfitting": overfit_lstm.loc[0, "interpretation"],
    },
])
display(summary.round(4))

rf_path = OUTPUT_DIR / "random_forest_h2s_fusion_colab.pkl"
scaler_lstm_path = OUTPUT_DIR / "lstm_feature_scaler_colab.pkl"
lstm_path = OUTPUT_DIR / "lstm_h2s_fusion_colab.h5"
metadata_path = OUTPUT_DIR / "metadata_entrainement_h2s_colab.json"
zip_path = COLAB_ROOT / "resultats_h2s_colab.zip"

joblib.dump(rf_pipeline, rf_path)
joblib.dump(feature_scaler, scaler_lstm_path)
lstm_model.save(lstm_path)

metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_path": str(DATASET_PATH),
    "gas": "H2S",
    "sensor_fusion": "C_H2S_fusion_ppm = mean(Sensor1, Sensor2, Sensor3, Sensor4)",
    "environment_features": [TEMPERATURE_COL, HUMIDITY_COL],
    "target": TARGET_COL,
    "balanced_rows_per_class_rf": BALANCE_ROWS_PER_CLASS,
    "lstm_parameters": {
        "window": LSTM_WINDOW,
        "horizon": LSTM_HORIZON,
        "stride": LSTM_STRIDE,
        "max_sequences_per_class": MAX_LSTM_SEQUENCES_PER_CLASS,
        "epochs_configured": LSTM_EPOCHS,
        "batch_size": LSTM_BATCH_SIZE,
        "architecture": "Bidirectional LSTM(64) + LSTM(32) + BatchNormalization + Dropout",
    },
    "rf_features": RF_FEATURES,
    "lstm_features": LSTM_FEATURES,
    "rf_metrics": {
        "train": rf_train_metrics,
        "validation": rf_val_metrics,
        "test": rf_test_metrics,
        "test_confusion_matrix": rf_test_cm.tolist(),
    },
    "lstm_metrics": {
        "train": lstm_train_metrics,
        "validation": lstm_val_metrics,
        "test": lstm_test_metrics,
        "test_confusion_matrix": lstm_test_cm.tolist(),
    },
}
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in [rf_path, scaler_lstm_path, lstm_path, metadata_path]:
        zf.write(path, arcname=path.name)

print("Fichiers sauvegardes dans :", OUTPUT_DIR)
print("Archive :", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Telechargement automatique disponible uniquement dans Google Colab.")
